<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-252-AI-and-Data-Engineering/blob/main/GI_Clinical_Decision_Support_%E2%80%94_full_RAG_pipeline_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GI Clinical Decision Support — full RAG pipeline (LangChain + Qwen3)

A retrieval-augmented generation pipeline for **gastroenterology clinical decision support** over clinical **notes and guidelines**, using a multi-billion-parameter open model (**Qwen3**) for detailed, grounded explanations.



What this builds:
- a biomedical embedding index over GI notes + guidelines (FAISS),
- Qwen3-8B loaded in 4-bit (fits a T4), thinking-mode off, low temperature for accuracy,
- a clinical prompt that forces grounded, **cited**, structured answers and **abstains** when the context is insufficient,
- an optional **QLoRA fine-tuning** section, and how fine-tuning + RAG combine.

Built for **current LangChain v1** . Safety: this is clinician decision support, not a prescriptive authority, and the corpus here is illustrative — see the final cell for production swaps.

In [5]:
# torch + CUDA are preinstalled in Colab — do NOT reinstall them (it can break CUDA).
# Qwen3 needs a recent transformers (>=4.51), so we upgrade it.
!pip install -q -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu
!pip install -q langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 151.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 132.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [6]:
import warnings, os, torch
warnings.filterwarnings("ignore")  # quiets the langchain-community sunset notice

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("⚠️  No GPU detected. Enable one: Runtime > Change runtime type > GPU. "
          "A 7-8B model is impractically slow on CPU.")
print("Device:", device)

# Optional Hugging Face token — only needed for *gated* models. Qwen3 is public (Apache-2.0),
# so this is not required; left in for convenience.
try:
    from google.colab import userdata
    tok = userdata.get("HUGGINGFACEHUB_API_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
        os.environ["HUGGINGFACEHUB_API_TOKEN"] = tok
except Exception:
    pass

Device: cuda


## Step 1 — Build the knowledge base (GI notes + guidelines)

⚠️ The text below is **illustrative and paraphrased** for a runnable demo — it is **not** a clinical reference. Each document carries `metadata['source']` so answers can cite it. In production, replace this with licensed ACG / AGA / AASLD guideline PDFs and de-identified institutional notes (see the final cell).

In [7]:
from langchain_core.documents import Document

_guidelines = [
    ("CRC screening guideline", "colorectal cancer screening",
     "Average-risk adults should begin colorectal cancer screening at age 45. Options "
     "include colonoscopy every 10 years or a stool-based test such as FIT at shorter "
     "intervals. Any positive non-colonoscopy test must be followed by a diagnostic "
     "colonoscopy. A first-degree relative with colorectal cancer usually means starting "
     "earlier and screening more often."),
    ("H. pylori guideline", "H. pylori",
     "When H. pylori is identified, eradication is indicated. Because clarithromycin "
     "resistance is common, bismuth quadruple therapy is a preferred first-line regimen. "
     "Confirm eradication at least four weeks after finishing antibiotics, holding acid "
     "suppression beforehand to avoid false-negative testing."),
    ("GERD guideline", "GERD",
     "For typical reflux symptoms without alarm features, an 8-week proton pump inhibitor "
     "trial with lifestyle measures is appropriate. Alarm features — dysphagia, "
     "odynophagia, weight loss, GI bleeding, anemia, or a mass — warrant prompt upper "
     "endoscopy rather than empiric therapy."),
    ("Barrett's esophagus guidance", "Barrett's esophagus",
     "Barrett's esophagus is a complication of chronic reflux, managed with surveillance "
     "endoscopy at intervals set by the degree of dysplasia. Confirmed dysplasia is "
     "generally referred for endoscopic eradication therapy."),
    ("IBD guideline", "inflammatory bowel disease",
     "Fecal calprotectin is a useful noninvasive marker of intestinal inflammation that "
     "helps separate inflammatory bowel disease from functional disorders and track "
     "disease activity. Diagnosis is confirmed with endoscopy and biopsy. Moderate-to-"
     "severe disease is commonly treated with biologics; corticosteroids are for short-"
     "term induction, not maintenance."),
    ("Acute pancreatitis guideline", "acute pancreatitis",
     "Acute pancreatitis is diagnosed when at least two of three criteria are present: "
     "characteristic epigastric pain, lipase or amylase over three times the upper limit "
     "of normal, or characteristic cross-sectional imaging. Early care includes goal-"
     "directed fluid resuscitation and analgesia plus identifying the cause; gallstone "
     "pancreatitis usually warrants cholecystectomy during the same admission once "
     "recovered."),
    ("GI bleeding / anemia guideline", "iron deficiency anemia",
     "New iron deficiency anemia in an adult — especially men and postmenopausal women — "
     "should prompt evaluation of the GI tract for a bleeding source, generally including "
     "colonoscopy and upper endoscopy."),
    ("Celiac disease guideline", "celiac disease",
     "Initial testing for celiac disease is serologic: tissue transglutaminase IgA with a "
     "total IgA level, performed while the patient is still eating gluten. A positive "
     "result is confirmed with duodenal biopsy before committing to a lifelong gluten-free "
     "diet."),
    ("Eosinophilic esophagitis guideline", "eosinophilic esophagitis",
     "Eosinophilic esophagitis should be considered with dysphagia or food impaction. "
     "Diagnosis requires esophageal biopsies showing an elevated eosinophil count. "
     "Treatment options include proton pump inhibitors, swallowed topical steroids, and "
     "dietary elimination."),
    ("IBS guidance", "irritable bowel syndrome",
     "Irritable bowel syndrome is a clinical diagnosis based on recurrent abdominal pain "
     "linked to defecation or a change in bowel habits, without alarm features. A low-"
     "FODMAP dietary trial and symptom-targeted therapy are reasonable first steps; alarm "
     "features prompt further workup."),
    ("HCC surveillance guidance", "hepatocellular carcinoma surveillance",
     "Patients with cirrhosis, and selected patients with chronic hepatitis B, should "
     "undergo hepatocellular carcinoma surveillance with abdominal ultrasound, with or "
     "without alpha-fetoprotein, every six months."),
    ("C. difficile guideline", "C. difficile infection",
     "Clostridioides difficile infection is confirmed with stool testing in patients with "
     "clinically significant diarrhea. Initial treatment is typically oral vancomycin or "
     "fidaxomicin; metronidazole is reserved for limited situations."),
    ("Diverticulitis guidance", "diverticulitis",
     "Uncomplicated acute diverticulitis can often be managed without antibiotics in "
     "selected immunocompetent patients with mild disease; complicated disease (abscess, "
     "perforation, obstruction) needs imaging-guided or surgical management. Colonoscopy "
     "is generally done after resolution if not recently performed."),
    ("Hepatitis B guidance", "hepatitis B",
     "Decisions to treat chronic hepatitis B depend on viral load, ALT level, and the "
     "presence of significant fibrosis or cirrhosis. Patients not yet meeting treatment "
     "criteria require ongoing monitoring."),
]

_notes = [
    ("Clinic note — reflux with alarm feature", "GERD",
     "65-year-old man with 6 months of heartburn now reports new difficulty swallowing "
     "solids and a 5 kg unintentional weight loss. No prior endoscopy. Taking intermittent "
     "antacids."),
    ("Clinic note — iron deficiency anemia", "iron deficiency anemia",
     "62-year-old man referred for fatigue; labs show hemoglobin 10.1 g/dL with low "
     "ferritin consistent with iron deficiency. No overt bleeding reported. No prior "
     "colonoscopy."),
    ("Clinic note — suspected IBD", "inflammatory bowel disease",
     "27-year-old with 3 months of bloody diarrhea and abdominal cramping; fecal "
     "calprotectin markedly elevated. No recent travel or antibiotics."),
]

documents = (
    [Document(page_content=t, metadata={"source": s, "topic": tp, "type": "guideline"})
     for s, tp, t in _guidelines]
    + [Document(page_content=t, metadata={"source": s, "topic": tp, "type": "note"})
       for s, tp, t in _notes]
)
print("✓ Built", len(documents), "documents")

✓ Built 17 documents


## Step 2 — Chunk (metadata preserved)

`RecursiveCharacterTextSplitter` keeps short guideline statements intact while splitting longer text on natural boundaries. Chunk metadata (including `source`) is carried through, which is what makes citation possible downstream.

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
chunks = splitter.split_documents(documents)
print("✓ Split into", len(chunks), "chunks")

✓ Split into 17 chunks


## Step 3 — Biomedical embeddings + FAISS index

A general embedding model (e.g. MiniLM) works, but a **biomedical** encoder retrieves clinical text far better. We default to a PubMedBERT-based sentence encoder and normalize embeddings (cosine-friendly).

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBED_ID = "pritamdeka/S-PubMedBert-MS-MARCO"   # biomedical retrieval embeddings
# Alternatives: "NeuML/pubmedbert-base-embeddings", "BAAI/bge-large-en-v1.5",
#               "Qwen/Qwen3-Embedding-0.6B"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_ID,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = FAISS.from_documents(chunks, embeddings)

# k = how many passages to ground each answer in. MMR option (commented) trades
# some relevance for diversity, useful when chunks are near-duplicates.
retriever = vector_store.as_retriever(search_kwargs={"k": 5})
# retriever = vector_store.as_retriever(search_type="mmr",
#                                       search_kwargs={"k": 5, "fetch_k": 15})
print("✓ Indexed", len(chunks), "chunks with", EMBED_ID)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Indexed 17 chunks with pritamdeka/S-PubMedBert-MS-MARCO


In [10]:
# Cross-encoder reranker: retrieve broadly, then keep only the best-matching passages.
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")   # "BAAI/bge-reranker-large" is stronger

def retrieve_reranked(question, fetch_k=8, top_n=3):
    candidates = vector_store.similarity_search(question, k=fetch_k)
    scores = reranker.predict([(question, d.page_content) for d in candidates])
    ranked = sorted(zip(candidates, scores), key=lambda ds: ds[1], reverse=True)
    return [d for d, _ in ranked[:top_n]]

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

## Step 4 — Load Qwen3-8B in 4-bit

4-bit quantization (QLoRA-style) lets an 8B model fit in a T4's 16 GB. Notes:
- **Thinking mode is disabled** so the model returns a clean clinical answer rather than `<think>…</think>` traces.
- **Low temperature (0.3)** favors faithful, reproducible clinical output; a small `repetition_penalty` prevents the looping Qwen3 can show at low temperature.
- **`max_new_tokens=1024`** allows the long, structured explanations requested.
- Swap `MODEL_ID` for a smaller/larger model depending on your GPU.

In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "Qwen/Qwen3-8B"          # fits a Colab T4 in 4-bit
# Alternatives:
#   "Qwen/Qwen3-4B-Instruct-2507"   -> lighter / faster
#   "Qwen/Qwen3-1.7B"               -> very light
#   "Qwen/Qwen3-14B" or "Qwen3-32B" -> stronger (needs A100 / more VRAM)
#   "Qwen/Qwen2.5-7B-Instruct"      -> rock-solid fallback, no thinking-mode handling

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # float16 for T4; bfloat16 on A100
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
)

pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=1024,            # room for detailed, structured explanations
    do_sample=True, temperature=0.3, top_p=0.8, top_k=20,
    repetition_penalty=1.1,         # guards against low-temp looping
    return_full_text=False,         # return only the completion, not the prompt
)
llm = HuggingFacePipeline(pipeline=pipe)
print("✓ LLM ready:", MODEL_ID)

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.9k [00:00<?, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✓ LLM ready: Qwen/Qwen3-8B


## Step 5 — Clinical prompt + grounded `ask()`

The system prompt forces three things that matter for clinical use: **answer only from retrieved context**, **abstain** when the context is insufficient, and return a **structured, cited** explanation. We apply Qwen3's chat template ourselves (with thinking disabled) and keep the source list for display.

In [12]:
SYSTEM_PROMPT = """You are a gastroenterology clinical decision-support assistant for licensed clinicians.
Answer using ONLY the information in the provided context passages.
If the context does not contain enough information to answer safely, say so explicitly and recommend
clinician review and consultation of primary sources — do not guess or rely on outside knowledge.

When the context is sufficient, give a thorough, well-structured answer with these sections:
1. Recommendation — a direct, concise answer.
2. Clinical rationale — detailed reasoning that explains the "why".
3. Supporting evidence — cite the specific source(s) you used, in brackets, e.g. [CRC screening guideline].
4. Caveats & red flags — contraindications, alarm features, exceptions, and patient factors that change management.
5. When to escalate / limits — what falls outside the provided guidance.

Cite sources inline as [source]. Do not state specific drug doses unless they appear in the context.
This output augments and does not replace clinical judgment."""

DISCLAIMER = ("\u26a0\ufe0f Decision support for licensed clinicians only \u2014 not a prescriptive "
              "authority. Verify against current guidelines and the patient's full record.")

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source', 'source')}] {d.page_content}" for d in docs)

def build_chat_prompt(question, context):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context passages:\n\n{context}\n\nClinician question: {question}"},
    ]
    # enable_thinking is a Qwen3 feature; fall back gracefully for other models.
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False,
                                             add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

import re

def cited_sources(answer_text, retrieved_sources):
    """Only the sources the model actually cited, limited to ones we retrieved
    (so any bracket text it invents doesn't appear as a 'source')."""
    cited = set(re.findall(r"\[([^\[\]]+)\]", answer_text))
    return sorted(retrieved_sources & cited), sorted(retrieved_sources - cited)

def ask(question, fetch_k=8, top_n=3):
    docs = retrieve_reranked(question, fetch_k=fetch_k, top_n=top_n)   # was: retriever.invoke(question)
    if not docs:
        return {"answer": "No relevant guideline or note was retrieved. Recommend clinician review.\n\n" + DISCLAIMER,
                "cited": [], "uncited": []}
    prompt = build_chat_prompt(question, format_docs(docs))
    answer = llm.invoke(prompt).strip()
    retrieved = {d.metadata.get("source", "source") for d in docs}
    used, uncited = cited_sources(answer, retrieved)
    return {"answer": answer + "\n\n" + DISCLAIMER, "cited": used, "uncited": uncited}

def show(question, **kw):
    r = ask(question, **kw)
    print("=" * 90); print("Q:", question); print("=" * 90)
    print(r["answer"])
    print("\nSources cited:", ", ".join(r["cited"]) or "(none)")
    if r["uncited"]:
        print("Also retrieved (not cited):", ", ".join(r["uncited"]))

# Optional stricter abstention using similarity scores (FAISS default = L2 distance,
# lower = closer; the right threshold depends on the embedding model — tune empirically):
#   scored = vector_store.similarity_search_with_score(question, k=k)
#   if not scored or scored[0][1] > SOME_THRESHOLD:
#       return abstain ...
print("✓ ask() ready")

✓ ask() ready


## Step 6 — Run example GI queries

The last query is deliberately **out of scope** for the corpus — it should make the model abstain rather than fabricate.

In [14]:
queries = [
    "A 45-year-old with no symptoms and no family history asks when to start colorectal cancer screening and what the options are.",
    "How should I manage a newly diagnosed H. pylori infection, and how do I confirm it has been eradicated?",
    "62-year-old man with new iron deficiency anemia and no overt bleeding — what is the appropriate GI workup?",
    "Patient with longstanding reflux now has new dysphagia and weight loss — what should I do?",
    "What is the recommended peri-procedural anticoagulation bridging protocol before an EGD?",  # out of scope -> abstain
]

for q in queries:
    show(q)
    print()

Q: A 45-year-old with no symptoms and no family history asks when to start colorectal cancer screening and what the options are.
1. **Recommendation**: The patient should begin colorectal cancer (CRC) screening at age 45, given their average risk status. Screening options include colonoscopy every 10 years or a stool-based test like FIT (Fecal Immunochemical Test) at shorter intervals [CRC screening guideline].

2. **Clinical rationale**: The American Cancer Society and other leading organizations have updated guidelines to recommend CRC screening initiation at age 45 for average-risk individuals due to rising incidence rates in younger populations. This recommendation applies to those without a personal or family history of CRC or polyps, which aligns with this patient’s profile. Colonoscopy remains the gold standard for definitive diagnosis and prevention, while stool-based tests offer non-invasive alternatives with lower adherence barriers [CRC screening guideline].

3. **Supporting

In [15]:
import time, textwrap, pandas as pd
from IPython.display import display, Markdown, HTML
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()   # silences the max_length / pipeline log lines

def ask_record(question):
    """Run one query and return structured fields (reuses your existing functions)."""
    t0 = time.time()
    docs = retrieve_reranked(question)
    body = generate(build_chat_prompt(question, format_docs(docs))).strip()
    retrieved = {d.metadata.get("source", "source") for d in docs}
    cited, uncited = cited_sources(body, retrieved)
    return {"question": question, "body": body, "cited": cited, "uncited": uncited,
            "abstained": len(cited) == 0, "secs": round(time.time() - t0, 1)}

def summary_table(rows):
    return pd.DataFrame([{
        "Question": textwrap.shorten(x["question"], 70),
        "Sources cited": ", ".join(x["cited"]) or "—",
        "# cited": len(x["cited"]),
        "Abstained": "✓" if x["abstained"] else "",
        "Time (s)": x["secs"],
    } for x in rows])

def show_results(queries):
    rows = [ask_record(q) for q in queries]
    display(HTML("<h3>Summary</h3>"))
    display(summary_table(rows))                      # scan everything at a glance
    display(HTML("<h3>Full answers</h3>"))
    for i, x in enumerate(rows, 1):
        footer = (f"\n\n---\n**Sources cited:** {', '.join(x['cited']) or '(none)'}  \n"
                  f"**Also retrieved (not cited):** {', '.join(x['uncited']) or '—'}  \n"
                  f"*Abstained: {'yes' if x['abstained'] else 'no'} · {x['secs']}s*\n\n{DISCLAIMER}")
        display(Markdown(f"### {i}. {x['question']}\n\n{x['body']}{footer}"))
        display(HTML("<hr style='border:0;border-top:1px solid #ddd'>"))
    return rows

In [18]:
import time, textwrap, pandas as pd, re
from IPython.display import display, Markdown, HTML
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

def ask_record(question):
    """Run one query and return structured fields."""
    t0 = time.time()

    # Retrieve
    docs = retrieve_reranked(question)
    if not docs:
        body = "No relevant guideline or note was retrieved. Recommend clinician review."
        cited, uncited = [], []
    else:
        # Generate
        prompt = build_chat_prompt(question, format_docs(docs))
        body = llm.invoke(prompt).strip()

        # Extract citations
        retrieved = {d.metadata.get("source", "source") for d in docs}
        cited, uncited = cited_sources(body, retrieved)

    return {
        "question": question,
        "body": body,
        "cited": cited,
        "uncited": uncited,
        "abstained": len(cited) == 0,
        "secs": round(time.time() - t0, 1),
    }

def summary_table(rows):
    """One-line summary per query."""
    return pd.DataFrame([{
        "Question": textwrap.shorten(x["question"], 70),
        "Sources cited": ", ".join(x["cited"]) or "—",
        "# cited": len(x["cited"]),
        "Abstained": "✓" if x["abstained"] else "",
        "Time (s)": x["secs"],
    } for x in rows])

def show_results(queries):
    """Run all queries and display summary table + full answers."""
    rows = [ask_record(q) for q in queries]

    # Summary table
    display(HTML("<h3>Summary</h3>"))
    display(summary_table(rows))

    # Full answers
    display(HTML("<h3>Full answers</h3>"))
    DISCLAIMER = ("⚠️ Decision support for licensed clinicians only — not a prescriptive "
                  "authority. Verify against current guidelines and the patient's full record.")
    for i, x in enumerate(rows, 1):
        footer = (f"\n\n---\n**Sources cited:** {', '.join(x['cited']) or '(none)'}  \n"
                  f"**Also retrieved (not cited):** {', '.join(x['uncited']) or '—'}  \n"
                  f"*Abstained: {'yes' if x['abstained'] else 'no'} · {x['secs']}s*\n\n{DISCLAIMER}")
        display(Markdown(f"### {i}. {x['question']}\n\n{x['body']}{footer}"))
        display(HTML("<hr style='border:0;border-top:1px solid #ddd'>"))

    return rows

In [19]:
rows = show_results(queries)

,Question,Sources cited,# cited,Abstained,Time (s)
0,A 45-year-old with no symptoms and no family h...,CRC screening guideline,1,,46.3
1,How should I manage a newly diagnosed H. pylor...,H. pylori guideline,1,,42.5
2,62-year-old man with new iron deficiency anemi...,GI bleeding / anemia guideline,1,,45.8
3,Patient with longstanding reflux now has new d...,"Eosinophilic esophagitis guideline, GERD guide...",2,,45.4
4,What is the recommended peri-procedural antico...,—,0,✓,32.8


### 1. A 45-year-old with no symptoms and no family history asks when to start colorectal cancer screening and what the options are.

1. **Recommendation**: The patient should begin colorectal cancer (CRC) screening at age 45, given their average risk status. Screening options include colonoscopy every 10 years or a stool-based test such as fecal immunochemical test (FIT) at shorter intervals [CRC screening guideline].

2. **Clinical rationale**: 
   - The American Cancer Society and other major guidelines have updated recommendations to begin CRC screening at age 45 for average-risk individuals, reflecting increased incidence rates in younger populations [CRC screening guideline].
   - Colonoscopy remains the gold standard for CRC screening due to its ability to detect and remove precancerous lesions during the procedure.
   - Stool-based tests like FIT are less invasive and can be performed more frequently, though they require follow-up colonoscopy if results are positive.

3. **Supporting evidence**:
   - The CRC screening guideline explicitly states that average-risk adults should begin screening at age 45 [CRC screening guideline].
   - It also outlines acceptable modalities for screening, including colonoscopy and stool-based testing [CRC screening guideline].

4. **Caveats & red flags**:
   - While the patient has no symptoms or family history, it is important to note that CRC screening is recommended for all average-risk adults beginning at age 45 regardless of personal or family history.
   - If the patient experiences changes in bowel habits, rectal bleeding, or unexplained weight loss, further investigation may be warranted.
   - Patients with a family history of CRC or other risk factors may need earlier or more frequent screening.

5. **When to escalate / limits**:
   - This recommendation applies specifically to average-risk individuals without additional risk factors.
   - For patients with a family history of CRC, inflammatory bowel disease, or other risk factors, individualized screening plans should be developed based on shared decision-making.
   - The provided guidance does not address patients with comorbid conditions, prior gastrointestinal surgery, or those who have had a previous diagnosis of adenomatous polyps. These cases require tailored approaches.

---
**Sources cited:** CRC screening guideline  
**Also retrieved (not cited):** GI bleeding / anemia guideline, HCC surveillance guidance  
*Abstained: no · 46.3s*

⚠️ Decision support for licensed clinicians only — not a prescriptive authority. Verify against current guidelines and the patient's full record.

### 2. How should I manage a newly diagnosed H. pylori infection, and how do I confirm it has been eradicated?

1. **Recommendation**: For a newly diagnosed *H. pylori* infection, initiate bismuth quadruple therapy as the preferred first-line regimen. Confirm eradication at least four weeks after completing antibiotic therapy through non-invasive testing, ensuring acid suppression is withheld prior to testing to prevent false-negative results [H. pylori guideline].

2. **Clinical rationale**: The *H. pylori* guideline emphasizes that eradication is necessary upon diagnosis due to its association with peptic ulcer disease, gastric cancer, and other complications. Clarithromycin resistance is prevalent, making bismuth quadruple therapy more effective than regimens containing clarithromycin. Eradication confirmation must occur at least four weeks post-treatment to allow for full clearance of the bacteria. Acid suppression should be avoided before testing to ensure accurate detection, as it may reduce the sensitivity of non-invasive tests like urea breath tests or stool antigen assays [H. pylori guideline].

3. **Supporting evidence**: The recommendation for bismuth quadruple therapy and the timing for eradication testing are directly supported by the *H. pylori* guideline [H. pylori guideline].

4. **Caveats & red flags**: 
   - Consider alternative therapies if the patient has a history of allergy to bismuth or other components of the regimen.
   - Be cautious in patients with renal impairment, as some components of bismuth quadruple therapy may require dose adjustment.
   - If eradication testing is inconclusive, consider repeat testing or endoscopic evaluation for persistent infection.

5. **When to escalate / limits**: 
   - Escalate care if there is no response to initial therapy or if eradication testing remains negative despite treatment.
   - Consult a specialist if the patient has complex comorbidities or if the infection is associated with severe complications such as gastric MALT lymphoma or refractory ulcers.

---
**Sources cited:** H. pylori guideline  
**Also retrieved (not cited):** Diverticulitis guidance, Hepatitis B guidance  
*Abstained: no · 42.5s*

⚠️ Decision support for licensed clinicians only — not a prescriptive authority. Verify against current guidelines and the patient's full record.

### 3. 62-year-old man with new iron deficiency anemia and no overt bleeding — what is the appropriate GI workup?

1. **Recommendation**: The appropriate GI workup for this 62-year-old man with new iron deficiency anemia and no overt bleeding includes colonoscopy and upper endoscopy [GI bleeding / anemia guideline].

2. **Clinical rationale**: Iron deficiency anemia in adults, particularly in men and postmenopausal women, is strongly associated with gastrointestinal (GI) blood loss. Given the absence of overt bleeding but presence of anemia, it is critical to evaluate for occult GI bleeding. Colonoscopy is essential to assess for colonic pathology such as polyps or malignancies, while upper endoscopy is necessary to rule out peptic ulcer disease, gastritis, or other upper GI sources of bleeding. This dual approach ensures comprehensive assessment of potential bleeding sites [GI bleeding / anemia guideline].

3. **Supporting evidence**: 
   - The GI bleeding/anemia guideline explicitly states that new iron deficiency anemia in adults warrants evaluation of the GI tract, typically involving both colonoscopy and upper endoscopy [GI bleeding / anemia guideline].

4. **Caveats & red flags**: 
   - While the patient reports no overt bleeding, the presence of anemia itself can be considered an alarm feature for GI bleeding, necessitating urgent investigation [GI bleeding / anemia guideline].
   - Patient age (62 years), male gender, and lack of prior colonoscopy further support the need for timely evaluation, given increased risk of colorectal cancer in this demographic [CRC screening guideline].

5. **When to escalate / limits**: 
   - If the patient presents with alarm features such as dysphagia, weight loss, or a family history of GI malignancy, escalation to more advanced imaging or referral to a gastroenterologist would be warranted.
   - In cases where the patient cannot undergo colonoscopy due to contraindications (e.g., coagulopathy, severe comorbidities), alternative strategies may be required, though this scenario is beyond the scope of the current guidelines [GI bleeding / anemia guideline].

---
**Sources cited:** GI bleeding / anemia guideline  
**Also retrieved (not cited):** Clinic note — iron deficiency anemia, GERD guideline  
*Abstained: no · 45.8s*

⚠️ Decision support for licensed clinicians only — not a prescriptive authority. Verify against current guidelines and the patient's full record.

### 4. Patient with longstanding reflux now has new dysphagia and weight loss — what should I do?

1. **Recommendation**: Prompt upper endoscopy is warranted due to the presence of alarm features (dysphagia and weight loss).

2. **Clinical rationale**: The emergence of new dysphagia and weight loss in a patient with longstanding reflux raises significant concern for complications such as Barrett’s esophagus, stricture formation, or even malignancy. These symptoms are classified as alarm features, which necessitate immediate diagnostic evaluation through upper endoscopy rather than proceeding with an empirical trial of proton pump inhibitors [GERD guideline].

3. **Supporting evidence**: 
   - The GERD guideline clearly states that patients presenting with alarm features such as dysphagia, weight loss, or gastrointestinal bleeding require prompt upper endoscopy instead of initial empiric therapy [GERD guideline].
   - The presence of dysphagia also aligns with the criteria for considering eosinophilic esophagitis, though this condition typically presents with additional symptoms like food impaction or a history of allergies [Eosinophilic esophagitis guideline].

4. **Caveats & red flags**:
   - While the guidelines emphasize the importance of evaluating alarm features, there may be rare cases where alternative diagnoses (e.g., strictures from prior reflux, or non-ulcer dyspepsia) could mimic these symptoms. A careful history and physical examination are essential to differentiate between benign and malignant processes.
   - Patients with a history of prior endoscopy or known Barrett’s esophagus may have different follow-up protocols depending on their level of dysplasia.

5. **When to escalate / limits**:
   - This recommendation applies specifically to patients with alarm features in the context of long-standing reflux. If the patient has no prior endoscopic evaluation or if there is uncertainty about the nature of the symptoms, further investigation beyond the initial endoscopy may be required.
   - The provided guidelines do not address complex comorbidities or rare presentations; thus, individualized assessment by a gastroenterologist is recommended.

---
**Sources cited:** Eosinophilic esophagitis guideline, GERD guideline  
**Also retrieved (not cited):** Barrett's esophagus guidance  
*Abstained: no · 45.4s*

⚠️ Decision support for licensed clinicians only — not a prescriptive authority. Verify against current guidelines and the patient's full record.

### 5. What is the recommended peri-procedural anticoagulation bridging protocol before an EGD?

The context provided does not include specific information about the recommended peri-procedural anticoagulation bridging protocol before an EGD (esophageal-gastric dilation). Therefore, it is not possible to provide a definitive recommendation based solely on the given context.

**Recommendation:**  
The recommended peri-procedural anticoagulation bridging protocol before an EGD is not specified in the provided context.

**Clinical rationale:**  
The context passages focus on acute pancreatitis, *H. pylori* infection, and GERD, but none address anticoagulation management or EGD-specific protocols. These guidelines emphasize diagnostic criteria, treatment approaches, and follow-up strategies, but do not cover procedural considerations such as anticoagulation bridging.

**Supporting evidence:**  
No specific source within the provided context addresses anticoagulation bridging before EGD.

**Caveats & red flags:**  
Anticoagulation management prior to procedures like EGD depends on several factors including the patient’s risk of thromboembolism, bleeding risk, and the type of anticoagulant being used. The absence of this information in the context means that individualized assessment by a clinician is necessary.

**When to escalate / limits:**  
This topic falls outside the scope of the provided context, which focuses on diagnosis and treatment of gastrointestinal conditions rather than procedural anticoagulation management. Clinicians should consult current guidelines or seek expert advice for personalized recommendations.

---
**Sources cited:** (none)  
**Also retrieved (not cited):** Acute pancreatitis guideline, GERD guideline, H. pylori guideline  
*Abstained: yes · 32.8s*

⚠️ Decision support for licensed clinicians only — not a prescriptive authority. Verify against current guidelines and the patient's full record.

## (Optional) Fine-tuning with QLoRA — and how it combines with RAG

**Use the right tool for each job.** RAG injects *facts, recency, and citations* at inference time. Fine-tuning shapes *format, tone, and behavior* — making the model reliably produce your structured clinical answer in your house style.

**Do not fine-tune to memorize guideline facts.** Baked-in facts go stale and the model will state them confidently even when wrong (the documented failure mode of guideline-specific LLMs). Keep facts in the retrievable corpus; fine-tune only the behavior. After fine-tuning, attach the adapter to the **same** RAG pipeline above — the two are complementary, not alternatives.

The cell below is a **template** (guarded by `RUN_FINETUNE`). Real fine-tuning needs hundreds-to-thousands of **clinician-reviewed** examples, and `trl` APIs change between versions — adjust to your installed version.

In [16]:
RUN_FINETUNE = False  # set True only with a curated, validated dataset + GPU time

if RUN_FINETUNE:
    !pip install -q -U peft trl datasets
    from peft import LoraConfig, PeftModel
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset

    # Instruction-style clinical Q&A in chat format. Shape only — needs real, reviewed data.
    examples = [
        {"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Context passages:\n\n[GERD guideline] ...\n\nClinician question: ..."},
            {"role": "assistant", "content": "1. Recommendation \u2014 ...\n2. Clinical rationale \u2014 ..."},
        ]},
        # ... many more ...
    ]
    ds = Dataset.from_list(examples)

    lora = LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    trainer = SFTTrainer(
        model=model,                       # the 4-bit base from Step 4 -> this is QLoRA
        train_dataset=ds,
        peft_config=lora,
        args=SFTConfig(
            output_dir="qwen3-gi-lora", num_train_epochs=1,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            learning_rate=2e-4, logging_steps=5, fp16=True, bf16=False,
        ),
    )
    trainer.train()
    trainer.save_model("qwen3-gi-lora")

    # To use the fine-tuned model in the RAG pipeline, reload base + adapter and rebuild pipe/llm:
    #   model = PeftModel.from_pretrained(model, "qwen3-gi-lora")
    #   pipe  = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=1024,
    #                    do_sample=True, temperature=0.3, top_p=0.8, repetition_penalty=1.1,
    #                    return_full_text=False)
    #   llm   = HuggingFacePipeline(pipeline=pipe)
    print("✓ Fine-tuning complete; adapter saved to qwen3-gi-lora/")
else:
    print("Fine-tuning skipped (RUN_FINETUNE = False).")

Fine-tuning skipped (RUN_FINETUNE = False).


## Going to production — swaps and safety

**Real corpus.** Replace the illustrative documents with licensed guideline PDFs and de-identified notes:
```python
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
docs = DirectoryLoader("guidelines/", glob="**/*.pdf", loader_cls=PyPDFLoader).load()
```
Tag each with rich metadata (society, year, section, evidence grade). **Recency matters most** — a missing or outdated guideline is the top cause of confidently wrong answers, so version and refresh the index.

**Retrieval quality.** Add a reranker (e.g. `BAAI/bge-reranker-large`) and consider hybrid (BM25 + dense) retrieval; chunking strategy materially affects accuracy.

**Notes / PHI.** Real notes require a PHI-compliant deployment and a validated de-identification step (clinical NER / Philter), not a regex — and patient data must not leak to uncontrolled third-party endpoints.

**Evaluation.** Pair automatic retrieval/answer metrics with a **clinician panel review** of factuality, completeness, and safety, plus adversarial probing and regression tests when the corpus changes.

**Safety / scope.** This is **clinician decision support, not a prescriptive authority**: every answer is traceable to cited sources, the system abstains when context is insufficient, and dosing is withheld unless present in the source. Clinical deployment also raises regulatory questions (clinical decision support vs. regulated software) — design so the clinician can independently verify every recommendation.